# Route Optimization

This notebook implements a flood-aware matatu route optimization system for Nairobi. When wards are predicted as flood-prone, the affected road segments are penalized using a flood-risk cost weight, and Dijkstra's algorithm is used to find safer alternative paths. The output is packaged as a GTFS-RT (General Transit Feed Specification — Realtime) feed, the industry-standard format consumable by transit apps such as Google Maps and Transit App.

### Pipeline
```
Ward flood probabilities (Random Forest model)
        ↓
Spatial join → map probabilities to OSMnx road edges
        ↓
Assign flood-risk cost: cost = travel_time × (1 + α × flood_probability)
        ↓
Weighted Dijkstra → find safest path per affected route
        ↓
GTFS-RT TripUpdate messages → production-ready feed
        ↓
Folium map → visualise original vs. recommended routes
```

## 1. Imports & Configuration

In [5]:
import sys
sys.path.append('..')

from Utils.feature_engineering import engineer_features, FEATS

import warnings
warnings.filterwarnings('ignore')

import pickle
import joblib
import json
from pathlib import Path
from sklearn.preprocessing import StandardScaler

import numpy as np
import pandas as pd
import geopandas as gpd
import networkx as nx
import osmnx as ox
import folium
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from shapely.geometry import Point, LineString

# GTFS-RT
from google.transit import gtfs_realtime_pb2

# Paths
BASE      = Path('')
DATA      = BASE / '../Data'
MODELS    = BASE / '../Models'
REPORTS   = BASE / 'Reports'
GTFS_DIR  = DATA / 'GTFS_FEED_2019'
REPORTS.mkdir(parents=True, exist_ok=True)

# Alpha — flood-risk cost multiplier
# Controls how aggressively the algorithm avoids flooded roads
# Low (1.0): slight detour preference | High (10.0): near-complete rerouting
ALPHA = 5.0

# Flood probability threshold above which a ward is considered high-risk
FLOOD_THRESHOLD = 0.45

# Nairobi bounding box for OSMnx road network download
NAIROBI_BBOX  = (-1.50, 36.60, -1.10, 37.10)  # (lat_min, lon_min, lat_max, lon_max)

WGS84  = 'EPSG:4326'
METRIC = 'EPSG:32737'

print('✓  Imports and configuration ready')

✓  Imports and configuration ready


## 2. Load Data

Load the ward boundaries, flood predictions from the saved Random Forest model, and the GTFS feed files.

In [6]:
# --- 2a. Ward boundaries & flood features ---
wards = gpd.read_file(DATA / 'floods.gpkg')
wards.columns = [c.lower().strip() for c in wards.columns]

# Clip to Nairobi
nairobi_wards = wards[wards['county'].str.lower().str.strip() == 'nairobi'].copy()
nairobi_wards = nairobi_wards.reset_index(drop=True)
print(f'✓  Nairobi wards loaded: {len(nairobi_wards)}')

# --- 2b. Load saved Random Forest model & generate flood probabilities ---
rf_model = joblib.load(MODELS / 'best_random_forest_model.joblib')

# Replicate the feature engineering from random_forest_notebook.ipynb
METRIC   = 'EPSG:32737'
LOW_ELEV = 1_620.0

engineered = engineer_features(nairobi_wards)
X = engineered[FEATS].fillna(engineered[FEATS].median()).to_numpy()

nairobi_wards['flood_prob'] = rf_model.predict_proba(X)[:, 1]
nairobi_wards['flood_risk'] = (nairobi_wards['flood_prob'] >= FLOOD_THRESHOLD).astype(int)

print(f'✓  Flood probabilities generated')
print(f'   High-risk wards: {nairobi_wards["flood_risk"].sum()} / {len(nairobi_wards)}')
print(f'   Probability range: {nairobi_wards["flood_prob"].min():.3f} - {nairobi_wards["flood_prob"].max():.3f}')

✓  Nairobi wards loaded: 85
✓  Flood probabilities generated
   High-risk wards: 34 / 85
   Probability range: 0.364 - 0.756


In [7]:
# --- 2c. Load GTFS feed ---
routes    = pd.read_csv(GTFS_DIR / 'routes.txt')
stops     = pd.read_csv(GTFS_DIR / 'stops.txt')
trips     = pd.read_csv(GTFS_DIR / 'trips.txt')
shapes    = pd.read_csv(GTFS_DIR / 'shapes.txt')
stop_times = pd.read_csv(GTFS_DIR / 'stop_times.txt')

# Convert stops to GeoDataFrame
stops_gdf = gpd.GeoDataFrame(
    stops,
    geometry=gpd.points_from_xy(stops['stop_lon'], stops['stop_lat']),
    crs=WGS84
)

# Convert shapes to GeoDataFrame (one LineString per shape_id)
shapes_gdf = (
    shapes.sort_values(['shape_id', 'shape_pt_sequence'])
    .groupby('shape_id')
    .apply(lambda g: LineString(zip(g['shape_pt_lon'], g['shape_pt_lat'])))
    .reset_index()
    .rename(columns={0: 'geometry'})
)
shapes_gdf = gpd.GeoDataFrame(shapes_gdf, geometry='geometry', crs=WGS84)

print(f'✓  GTFS loaded')
print(f'   Routes: {len(routes)} | Stops: {len(stops)} | Shapes: {len(shapes_gdf)}')

✓  GTFS loaded
   Routes: 136 | Stops: 4284 | Shapes: 272


## 3. Download Nairobi Road Network

OSMnx downloads the real Nairobi road network from OpenStreetMap as a directed graph. Each edge represents a road segment and carries attributes including length and speed, from which travel time is computed.

In [ ]:
print('Downloading Nairobi road network from OpenStreetMap...')

# Download drivable road network within Nairobi bounding box
G = ox.graph_from_bbox(
    bbox=NAIROBI_BBOX,
    network_type='drive',
    simplify=True
)

# Add travel time (seconds) to each edge based on length and speed
G = ox.add_edge_speeds(G)
G = ox.add_edge_travel_times(G)

print(f'✓  Road network downloaded')
print(f'   Nodes: {G.number_of_nodes():,} | Edges: {G.number_of_edges():,}')

## 4. Map Flood Probabilities to Road Edges

Each road edge in the OSMnx graph is assigned the flood probability of the ward it falls within. Edges outside any ward boundary (unlikely but possible at the periphery) default to zero flood probability.

In [ ]:
# Convert edges to GeoDataFrame
edges_gdf = ox.graph_to_gdfs(G, nodes=False, edges=True)
edges_gdf = edges_gdf.reset_index()

# Compute edge midpoints for spatial join
edges_gdf['midpoint'] = edges_gdf['geometry'].interpolate(0.5, normalized=True)
edges_mid = gpd.GeoDataFrame(edges_gdf[['u', 'v', 'key', 'midpoint']],
                              geometry='midpoint', crs=WGS84)

# Spatial join: assign ward flood probability to each edge
edges_joined = gpd.sjoin(
    edges_mid,
    nairobi_wards[['flood_prob', 'ward', 'geometry']],
    how='left',
    predicate='within'
)

# Handle duplicates (edge midpoint may fall in multiple wards at boundary)
edges_joined = edges_joined.groupby(['u', 'v', 'key'])['flood_prob'].max().reset_index()
edges_joined['flood_prob'] = edges_joined['flood_prob'].fillna(0.0)

# Build a lookup dictionary: (u, v, key) → flood_prob
flood_prob_map = {
    (row['u'], row['v'], row['key']): row['flood_prob']
    for _, row in edges_joined.iterrows()
}

flooded_edges = (edges_joined['flood_prob'] >= FLOOD_THRESHOLD).sum()
print(f'✓  Flood probabilities mapped to road edges')
print(f'   Total edges: {len(edges_joined):,}')
print(f'   High-risk edges (prob ≥ {FLOOD_THRESHOLD}): {flooded_edges:,}')

## 5. Build Flood-Weighted Graph

Each edge's cost is modified from raw travel time to a flood-penalized travel time:

$$\text{cost} = \text{travel\_time} \times (1 + \alpha \times \text{flood\_probability})$$

Where $\alpha$ (alpha) controls the strength of the penalty. A value of 5.0 means a road with 100% flood probability costs 6× more than normal — the algorithm strongly prefers alternatives, but will still use the flooded road if there is truly no other path.

In [ ]:
# Add flood-weighted cost to every edge in the graph
for u, v, key, data in G.edges(keys=True, data=True):
    travel_time  = data.get('travel_time', 60)  # default 60s if missing
    flood_prob   = flood_prob_map.get((u, v, key), 0.0)
    data['flood_cost'] = travel_time * (1 + ALPHA * flood_prob)

print(f'✓  Flood-weighted costs assigned to all edges (α = {ALPHA})')

# Sanity check: compare normal vs flood-weighted cost for a high-risk edge
sample = [
    (u, v, k, d) for u, v, k, d in G.edges(keys=True, data=True)
    if flood_prob_map.get((u, v, k), 0) >= FLOOD_THRESHOLD
]
if sample:
    u, v, k, d = sample[0]
    print(f'   Example high-risk edge ({u} → {v}):')
    print(f'     Travel time  : {d["travel_time"]:.1f}s')
    print(f'     Flood prob   : {flood_prob_map.get((u, v, k), 0):.3f}')
    print(f'     Flood cost   : {d["flood_cost"]:.1f}s')

## 6. Identify Affected Routes & Run Weighted Dijkstra

For each GTFS route, check whether any of its stops fall within a high-risk ward. If so, find the safest alternative path between the route's terminal stops using Dijkstra on the flood-weighted graph.

In [ ]:
# Spatial join: which stops are in high-risk wards?
stops_joined = gpd.sjoin(
    stops_gdf,
    nairobi_wards[nairobi_wards['flood_risk'] == 1][['ward', 'flood_prob', 'geometry']],
    how='left',
    predicate='within'
)
affected_stops = set(stops_joined[stops_joined['flood_prob'].notna()]['stop_id'].tolist())
print(f'✓  Affected stops identified: {len(affected_stops)} / {len(stops)}')

# Find routes that serve at least one affected stop
affected_trip_ids = stop_times[stop_times['stop_id'].isin(affected_stops)]['trip_id'].unique()
affected_route_ids = trips[trips['trip_id'].isin(affected_trip_ids)]['route_id'].unique()
print(f'   Affected routes: {len(affected_route_ids)} / {len(routes)}')

In [ ]:
def get_route_terminals(route_id: str) -> tuple:
    """Get the first and last stop coordinates for a given route."""
    trip_id = trips[trips['route_id'] == route_id]['trip_id'].iloc[0]
    route_stops = (
        stop_times[stop_times['trip_id'] == trip_id]
        .sort_values('stop_sequence')
        .merge(stops, on='stop_id')
    )
    origin = route_stops.iloc[0]
    destination = route_stops.iloc[-1]
    return (
        (origin['stop_lat'], origin['stop_lon'], origin['stop_name']),
        (destination['stop_lat'], destination['stop_lon'], destination['stop_name'])
    )


def find_alternative_route(route_id: str, G: nx.MultiDiGraph) -> dict | None:
    """Find the flood-cost-minimizing path between a route's terminal stops."""
    try:
        origin, destination = get_route_terminals(route_id)

        # Snap terminal coordinates to nearest OSMnx nodes
        orig_node = ox.nearest_nodes(G, X=origin[1],    Y=origin[0])
        dest_node = ox.nearest_nodes(G, X=destination[1], Y=destination[0])

        # Shortest path on normal travel time (original)
        original_path = nx.shortest_path(
            G, orig_node, dest_node, weight='travel_time'
        )

        # Shortest path on flood-weighted cost (alternative)
        alternative_path = nx.shortest_path(
            G, orig_node, dest_node, weight='flood_cost'
        )

        # Compute metrics for both paths
        def path_metrics(path, cost_attr):
            total_cost = sum(
                G[path[i]][path[i+1]][0].get(cost_attr, 0)
                for i in range(len(path) - 1)
            )
            total_time = sum(
                G[path[i]][path[i+1]][0].get('travel_time', 0)
                for i in range(len(path) - 1)
            )
            avg_flood_prob = np.mean([
                flood_prob_map.get((path[i], path[i+1], 0), 0.0)
                for i in range(len(path) - 1)
            ])
            return total_cost, total_time, avg_flood_prob

        orig_cost, orig_time, orig_flood = path_metrics(original_path, 'flood_cost')
        alt_cost,  alt_time,  alt_flood  = path_metrics(alternative_path, 'flood_cost')

        return {
            'route_id':         route_id,
            'origin':           origin[2],
            'destination':      destination[2],
            'original_path':    original_path,
            'alternative_path': alternative_path,
            'original_time_s':  orig_time,
            'alternative_time_s': alt_time,
            'extra_time_min':   round((alt_time - orig_time) / 60, 1),
            'original_flood_prob':    round(orig_flood, 3),
            'alternative_flood_prob': round(alt_flood, 3),
            'risk_reduction':   round(orig_flood - alt_flood, 3),
        }

    except Exception as e:
        return None


# Run for all affected routes
print(f'Running weighted Dijkstra for {len(affected_route_ids)} affected routes...')
rerouting_results = []

for i, route_id in enumerate(affected_route_ids):
    result = find_alternative_route(route_id, G)
    if result:
        rerouting_results.append(result)
    if (i + 1) % 10 == 0:
        print(f'  Processed {i+1}/{len(affected_route_ids)} routes...')

results_df = pd.DataFrame(rerouting_results)
print(f'\n✓  Rerouting complete: {len(results_df)} routes successfully rerouted')

## 7. Rerouting Summary

The table below summarises the rerouting results — showing the tradeoff between extra travel time and flood risk reduction for each affected route.

In [ ]:
# Summary table
summary = results_df[[
    'route_id', 'origin', 'destination',
    'original_flood_prob', 'alternative_flood_prob',
    'risk_reduction', 'extra_time_min'
]].copy()

summary.columns = [
    'Route ID', 'Origin', 'Destination',
    'Original Flood Risk', 'Alternative Flood Risk',
    'Risk Reduction', 'Extra Travel Time (min)'
]

summary = summary.sort_values('Risk Reduction', ascending=False)

print('Top 10 most improved routes:')
print(summary.head(10).to_string(index=False))

print(f'\nOverall statistics:')
print(f'  Average risk reduction  : {results_df["risk_reduction"].mean():.3f}')
print(f'  Average extra time (min): {results_df["extra_time_min"].mean():.1f}')
print(f'  Routes with same path   : {(results_df["risk_reduction"] == 0).sum()} (no alternative found)')

In [ ]:
# Visualise tradeoff: risk reduction vs extra travel time
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter: risk reduction vs extra time
ax = axes[0]
ax.scatter(
    results_df['extra_time_min'],
    results_df['risk_reduction'],
    alpha=0.7, color='#1D9E75', edgecolors='white', s=60
)
ax.axhline(0, color='gray', linestyle='--', lw=1)
ax.axvline(0, color='gray', linestyle='--', lw=1)
ax.set_xlabel('Extra Travel Time (minutes)')
ax.set_ylabel('Flood Risk Reduction')
ax.set_title('Risk Reduction vs. Extra Travel Time\nper Rerouted Matatu Route')

# Bar: distribution of extra travel time
ax = axes[1]
ax.hist(results_df['extra_time_min'], bins=20, color='#378ADD', edgecolor='white', alpha=0.85)
ax.axvline(results_df['extra_time_min'].mean(), color='#E24B4A',
           linestyle='--', lw=2, label=f'Mean: {results_df["extra_time_min"].mean():.1f} min')
ax.set_xlabel('Extra Travel Time (minutes)')
ax.set_ylabel('Number of Routes')
ax.set_title('Distribution of Extra Travel Time\nfor Alternative Routes')
ax.legend()

plt.tight_layout()
plt.savefig(REPORTS / 'rerouting_tradeoff.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓  Tradeoff chart saved')

## 8. Generate GTFS-RT Feed

The rerouting decisions are packaged as a GTFS-RT (General Transit Feed Specification — Realtime) `FeedMessage`. This is the industry-standard protobuf format used by transit apps including Google Maps and Transit App. Each affected trip receives a `TripUpdate` message with `StopTimeUpdate` entries flagging affected stops, and the alternative route details are embedded as a JSON extension.

In a production system this feed would be served over HTTP and refreshed every 30–60 seconds as flood conditions evolve.

In [ ]:
import time

def build_gtfs_rt_feed(results: list[dict], G: nx.MultiDiGraph) -> gtfs_realtime_pb2.FeedMessage:
    """Build a GTFS-RT FeedMessage from rerouting results."""
    feed = gtfs_realtime_pb2.FeedMessage()

    # Feed header
    feed.header.gtfs_realtime_version = '2.0'
    feed.header.incrementality = gtfs_realtime_pb2.FeedHeader.FULL_DATASET
    feed.header.timestamp = int(time.time())

    for result in results:
        route_id = result['route_id']

        # Get all trips for this route
        route_trips = trips[trips['route_id'] == route_id]['trip_id'].tolist()

        for trip_id in route_trips:
            entity = feed.entity.add()
            entity.id = f'reroute_{trip_id}'

            # Trip descriptor
            entity.trip_update.trip.trip_id    = str(trip_id)
            entity.trip_update.trip.route_id   = str(route_id)
            entity.trip_update.trip.schedule_relationship = (
                gtfs_realtime_pb2.TripDescriptor.ADDED  # new alternative trip
            )

            # Get stops for this trip
            trip_stop_times = (
                stop_times[stop_times['trip_id'] == trip_id]
                .sort_values('stop_sequence')
            )

            # Add StopTimeUpdate for each stop — flag affected stops
            for _, st_row in trip_stop_times.iterrows():
                stop_update = entity.trip_update.stop_time_update.add()
                stop_update.stop_sequence = int(st_row['stop_sequence'])
                stop_update.stop_id       = str(st_row['stop_id'])

                if st_row['stop_id'] in affected_stops:
                    # Flag affected stops as skipped
                    stop_update.schedule_relationship = (
                        gtfs_realtime_pb2.TripUpdate.StopTimeUpdate.SKIPPED
                    )
                else:
                    stop_update.schedule_relationship = (
                        gtfs_realtime_pb2.TripUpdate.StopTimeUpdate.SCHEDULED
                    )

    return feed


# Build and save the feed
feed = build_gtfs_rt_feed(rerouting_results, G)

feed_path = REPORTS / 'flood_rerouting_feed.pb'
with open(feed_path, 'wb') as f:
    f.write(feed.SerializeToString())

print(f'✓  GTFS-RT feed generated')
print(f'   Entities (trip updates): {len(feed.entity)}')
print(f'   Feed saved to: {feed_path}')
print(f'   Feed size: {feed_path.stat().st_size / 1024:.1f} KB')

## 9. Visualise on Folium Map

The interactive map below shows:
- **Ward flood risk** — colour-coded from low (green) to critical (red)
- **Original route** — blue line showing the standard matatu path
- **Alternative route** — orange line showing the flood-avoiding path
- **Affected stops** — red markers at stops within high-risk wards

In [ ]:
def flood_color(prob: float) -> str:
    if prob >= 0.70: return '#E24B4A'   # critical
    if prob >= 0.45: return '#EF9F27'   # high
    if prob >= 0.20: return '#F9CB42'   # moderate
    return '#1D9E75'                    # low


def path_to_linestring(path: list, G: nx.MultiDiGraph) -> list:
    """Convert an OSMnx node path to a list of [lat, lon] coordinates."""
    coords = []
    for node in path:
        node_data = G.nodes[node]
        coords.append([node_data['y'], node_data['x']])
    return coords


# Map centred on Nairobi
centre = [
    (NAIROBI_BBOX[0] + NAIROBI_BBOX[2]) / 2,
    (NAIROBI_BBOX[1] + NAIROBI_BBOX[3]) / 2
]
fmap = folium.Map(location=centre, zoom_start=12, tiles='CartoDB dark_matter')

# --- Ward flood risk layer ---
for _, ward in nairobi_wards.iterrows():
    if ward.geometry is None:
        continue
    prob  = float(ward['flood_prob'])
    color = flood_color(prob)
    folium.GeoJson(
        ward.geometry.__geo_interface__,
        style_function=lambda _, c=color: {
            'fillColor':   c,
            'fillOpacity': 0.40,
            'color':       c,
            'weight':      1.0,
        },
        tooltip=f"{ward['ward']} | Flood prob: {prob:.1%}",
    ).add_to(fmap)

# --- Affected stops ---
for _, stop in stops_gdf[stops_gdf['stop_id'].isin(affected_stops)].iterrows():
    folium.CircleMarker(
        location=[stop.geometry.y, stop.geometry.x],
        radius=4,
        color='#E24B4A',
        fill=True,
        fill_opacity=0.9,
        tooltip=f"⚠ Affected stop: {stop.get('stop_name', stop['stop_id'])}"
    ).add_to(fmap)

# --- Original vs alternative routes (first 5 for clarity) ---
for result in rerouting_results[:5]:
    orig_coords = path_to_linestring(result['original_path'], G)
    alt_coords  = path_to_linestring(result['alternative_path'], G)

    # Original route — blue
    folium.PolyLine(
        orig_coords,
        color='#378ADD', weight=3, opacity=0.7,
        tooltip=f"Route {result['route_id']} — Original"
    ).add_to(fmap)

    # Alternative route — orange
    folium.PolyLine(
        alt_coords,
        color='#EF9F27', weight=3, opacity=0.9, dash_array='8',
        tooltip=(
            f"Route {result['route_id']} — Alternative\n"
            f"Extra time: +{result['extra_time_min']} min | "
            f"Risk reduction: {result['risk_reduction']:.3f}"
        )
    ).add_to(fmap)

map_path = REPORTS / 'flood_rerouting_map.html'
fmap.save(str(map_path))
print(f'✓  Interactive map saved → {map_path}')
fmap

## 10. Save Results

Save the rerouting summary as a CSV for reporting, and export the GTFS-RT feed path for integration with downstream transit systems.

In [ ]:
# Save summary table (drop path lists — not CSV-friendly)
csv_cols = [
    'route_id', 'origin', 'destination',
    'original_flood_prob', 'alternative_flood_prob',
    'risk_reduction', 'original_time_s', 'alternative_time_s', 'extra_time_min'
]
results_df[csv_cols].to_csv(REPORTS / 'rerouting_summary.csv', index=False)

# Save metadata
metadata = {
    'alpha':             ALPHA,
    'flood_threshold':   FLOOD_THRESHOLD,
    'total_routes':      len(routes),
    'affected_routes':   len(affected_route_ids),
    'rerouted_routes':   len(results_df),
    'affected_stops':    len(affected_stops),
    'avg_risk_reduction': round(float(results_df['risk_reduction'].mean()), 4),
    'avg_extra_time_min': round(float(results_df['extra_time_min'].mean()), 2),
    'gtfs_rt_feed':      str(REPORTS / 'flood_rerouting_feed.pb'),
}

with open(REPORTS / 'rerouting_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print('✓  All outputs saved:')
print(f'   Reports/rerouting_summary.csv')
print(f'   Reports/rerouting_metadata.json')
print(f'   Reports/flood_rerouting_feed.pb')
print(f'   Reports/flood_rerouting_map.html')
print(f'   Reports/rerouting_tradeoff.png')
print(f'\nMetadata:')
print(json.dumps(metadata, indent=2))